# ==============================================================================
# PROJECT: EXPLORATORY DATA ANALYSIS (EDA) - SPOTIFY TOP SONGS 2023
# DESCRIPTION: Full Report
# DATASET: nelgiriyewithana/top-spotify-songs-2023
# ==============================================================================


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import kagglehub
import os
import warnings
warnings.filterwarnings('ignore')

print("🎧 Spotify Top Songs 2023 - Full EDA Report")

🎧 Spotify Top Songs 2023 - Full EDA Report


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình mặc định cho biểu đồ
plt.style.use('ggplot')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 16
print("✅ Đã cấu hình Matplotlib & Seaborn")

✅ Đã cấu hình Matplotlib & Seaborn


In [3]:
import plotly.io as pio
from pathlib import Path

OUTPUT_JSON_DIR = Path('eda_json')
OUTPUT_JSON_DIR.mkdir(exist_ok=True)

def save_div_to_json(fig, name: str, show: bool = True):
    # fig.update_layout(title=None) # Bạn có thể bỏ title ở đây để UI gọn hơn
    path = OUTPUT_JSON_DIR / f'{name}.json'

    # Xuất thành chuỗi JSON
    json_data = pio.to_json(fig)
    path.write_text(json_data, encoding='utf-8')
    print(f'[saved JSON] {path}')

# --- STAGE 1: DATA ACQUISITION & INITIAL INSPECTION ---


In [4]:
print("--- STAGE 1: DATA ACQUISITION ---")
path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df_raw = pd.read_csv(os.path.join(path, csv_file), encoding='latin-1')

print(f"Dataset loaded successfully: {df_raw.shape[0]:,} samples, {df_raw.shape[1]} features.")

# Dataset Overview
print("\n📊 DATASET OVERVIEW")
print(f"Training Samples : {len(df_raw):,}")
print(f"Total Features   : {df_raw.shape[1]}")
print(f"Numerical        : {df_raw.select_dtypes(include=np.number).shape[1]}")
print(f"Categorical      : {df_raw.select_dtypes(exclude=np.number).shape[1]}")

df_raw.info()
display(df_raw.head(10))

--- STAGE 1: DATA ACQUISITION ---


100%|██████████| 47.1k/47.1k [00:00<00:00, 35.0MB/s]

Extracting files...
Dataset loaded successfully: 953 samples, 24 features.

📊 DATASET OVERVIEW
Training Samples : 953
Total Features   : 24
Numerical        : 17
Categorical      : 7
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   track_name            953 non-null    object
 1   artist(s)_name        953 non-null    object
 2   artist_count          953 non-null    int64 
 3   released_year         953 non-null    int64 
 4   released_month        953 non-null    int64 
 5   released_day          953 non-null    int64 
 6   in_spotify_playlists  953 non-null    int64 
 7   in_spotify_charts     953 non-null    int64 
 8   streams               953 non-null    object
 9   in_apple_playlists    953 non-null    int64 
 10  in_apple_charts       953 non-null    int64 
 11  in_deezer_playlists   953 non-null    object
 12  in_deez

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6
5,Sprinter,"Dave, Central Cee",2,2023,6,1,2186,91,183706234,67,...,141,C#,Major,92,66,58,19,0,8,24
6,Ella Baila Sola,"Eslabon Armado, Peso Pluma",2,2023,3,16,3090,50,725980112,34,...,148,F,Minor,67,83,76,48,0,8,3
7,Columbia,Quevedo,1,2023,7,7,714,43,58149378,25,...,100,F,Major,67,26,71,37,0,11,4
8,fukumean,Gunna,1,2023,5,15,1096,83,95217315,60,...,130,C#,Minor,85,22,62,12,0,28,9
9,La Bebe - Remix,"Peso Pluma, Yng Lvcas",2,2023,3,17,2953,44,553634067,49,...,170,D,Minor,81,56,48,21,0,8,33


# --- STAGE 2: DATA PREPROCESSING & NOISE DETECTION ---


In [5]:
print("--- STAGE 2: DATA PREPROCESSING ---")
df_cleaned = df_raw.copy()

# 2.1 Clean structural noise in 'streams'
is_noise = pd.to_numeric(df_cleaned['streams'], errors='coerce').isna()
if is_noise.sum() > 0:
    print(f"Found {is_noise.sum()} corrupted record(s) in 'streams' → Dropped")
df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
df_cleaned = df_cleaned.dropna(subset=['streams'])
df_cleaned['streams'] = df_cleaned['streams'].astype(np.int64)

# 2.2 Clean comma in chart columns
platform_cols = ['in_spotify_playlists','in_spotify_charts','in_apple_playlists',
                 'in_apple_charts','in_deezer_playlists','in_deezer_charts','in_shazam_charts']
for col in platform_cols:
    if df_cleaned[col].dtype == 'object':
        df_cleaned[col] = df_cleaned[col].astype(str).str.replace(',', '')
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

# 2.3 Missing values
df_cleaned['key'] = df_cleaned['key'].fillna('Unknown')
df_cleaned['in_shazam_charts'] = df_cleaned['in_shazam_charts'].fillna(0)

# 2.4 Datetime + weekday
df_cleaned['release_date'] = pd.to_datetime(
    df_cleaned[['released_year','released_month','released_day']].astype(str).agg('-'.join, axis=1)
)
df_cleaned['release_day_name'] = df_cleaned['release_date'].dt.day_name()

# 2.5 Feature classification
exclude = ['track_name', 'artist(s)_name', 'release_date']
numeric_cols = df_cleaned.select_dtypes(include=np.number).columns.difference(exclude).tolist()
categorical_cols = df_cleaned.select_dtypes(exclude=np.number).columns.difference(exclude).tolist()

print(f"✅ Classified {len(numeric_cols)} numerical + {len(categorical_cols)} categorical features")

--- STAGE 2: DATA PREPROCESSING ---
Found 1 corrupted record(s) in 'streams' → Dropped
✅ Classified 20 numerical + 3 categorical features


In [6]:
print("📋 FEATURE DESCRIPTIONS")
feature_desc = pd.DataFrame({
    'Feature': ['track_name', 'artist(s)_name', 'artist_count', 'released_year', 'released_month',
                'released_day', 'in_spotify_playlists', 'in_spotify_charts', 'streams',
                'in_apple_playlists', 'in_apple_charts', 'in_deezer_playlists', 'in_deezer_charts',
                'in_shazam_charts', 'bpm', 'key', 'mode', 'danceability_%', 'valence_%',
                'energy_%', 'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%'],
    'Type': ['Categorical', 'Categorical', 'Numerical', 'Numerical', 'Numerical',
             'Numerical', 'Numerical', 'Numerical', 'Numerical', 'Numerical', 'Numerical',
             'Numerical', 'Numerical', 'Numerical', 'Numerical', 'Categorical', 'Categorical',
             'Numerical', 'Numerical', 'Numerical', 'Numerical', 'Numerical', 'Numerical', 'Numerical'],
    'Description': ['Name of the song', 'Name of the artist(s) of the song', 'Number of artists contributing to the song', 'Year when the song was released', 'Month when the song was released',
                    'Day of the month when the song was released', 'Number of Spotify playlists the song is included in', 'Presence and rank of the song on Spotify charts', 'Total number of streams on Spotify',
                    'Number of Apple Music playlists the song is included in', 'Presence and rank of the song on Apple Music charts', 'Number of Deezer playlists the song is included in', 'Presence and rank of the song on Deezer charts',
                    'Presence and rank of the song on Shazam charts', 'Beats per minute, a measure of song tempo', 'Key of the song', 'Mode of the song (major or minor)', 'Percentage indicating how suitable the song is for dancing',
                    'Positivity of the song\'s musical content', 'Perceived energy level of the song', 'Amount of acoustic sound in the song', 'Amount of instrumental content in the song', 'Presence of live performance elements', 'Amount of spoken words in the song']
})
display(feature_desc)

📋 FEATURE DESCRIPTIONS


,Feature,Type,Description
0,track_name,Categorical,Name of the song
1,artist(s)_name,Categorical,Name of the artist(s) of the song
2,artist_count,Numerical,Number of artists contributing to the song
3,released_year,Numerical,Year when the song was released
4,released_month,Numerical,Month when the song was released
5,released_day,Numerical,Day of the month when the song was released
6,in_spotify_playlists,Numerical,Number of Spotify playlists the song is includ...
7,in_spotify_charts,Numerical,Presence and rank of the song on Spotify charts
8,streams,Numerical,Total number of streams on Spotify
9,in_apple_playlists,Numerical,Number of Apple Music playlists the song is in...


In [7]:
print("⭐ TOP 10 SONGS FOR EACH NUMERICAL FEATURE ⭐")

# Ensure this runs after Stage 2 where numeric_cols and df_cleaned are defined
try:
    performance_numeric_cols = [col for col in numeric_cols if col not in ['released_year', 'released_month', 'released_day', 'artist_count']]

    for col in performance_numeric_cols:
        print(f"\n--- Top 10 Songs by {col.replace('_', ' ').title()} ---")
        top_songs = df_cleaned.nlargest(10, col)[['track_name', 'artist(s)_name', col, 'streams']]
        display(top_songs)
except NameError:
    print("❌ Lỗi: Biến 'numeric_cols' chưa được định nghĩa. Vui lòng chạy cell 1190ecb2 (Stage 2) trước.")

⭐ TOP 10 SONGS FOR EACH NUMERICAL FEATURE ⭐

--- Top 10 Songs by Acousticness % ---


,track_name,artist(s)_name,acousticness_%,streams
167,The Night We Met,Lord Huron,97,1410088830
940,Sweet Nothing,Taylor Swift,97,186104310
17,What Was I Made For? [From The Motion Picture ...,Billie Eilish,96,30546883
575,LA FAMA (with The Weeknd),"The Weeknd, ROSALï¿½",95,374706940
169,When I Was Your Man,Bruno Mars,94,1661187319
858,Boyfriends,Harry Styles,94,137070925
128,lovely - Bonus Track,"Billie Eilish, Khalid",93,2355719893
587,Miserable Man,David Kushner,93,124407432
623,All of Me,John Legend,92,2086124197
652,The Joker And The Queen (feat. Taylor Swift),"Ed Sheeran, Taylor Swift",92,146789379



--- Top 10 Songs by Bpm ---


,track_name,artist(s)_name,bpm,streams
100,Lover,Taylor Swift,206,882831184
506,We Don't Talk About Bruno,"Adassa, Mauro Castillo, Stephanie Beatriz, Enc...",206,432719968
28,Last Night,Morgan Wallen,204,429829812
244,Until I Found You,Stephen Sanchez,202,726434358
447,It's the Most Wonderful Time of the Year,Andy Williams,202,663832097
726,O.O,NMIXX,200,135444283
67,People,Libianca,198,373199958
775,La Corriente,"Tony Dize, Bad Bunny",196,479655659
155,Un Finde | CROSSOVER #2,"Big One, FMK, Ke personajes",192,142095275
710,Chale,Eden Muï¿½ï,189,299648208



--- Top 10 Songs by Danceability % ---


...

,track_name,artist(s)_name,speechiness_%,streams
247,Cartï¿½ï¿½o B,"MC Caverinha, KayBlack",64,71573339
935,On BS,"Drake, 21 Savage",59,170413877
209,Area Codes,"Kaliii, Kaliii",49,113509496
426,Limbo,Freddie Dredd,46,199386237
780,Savior,"Kendrick Lamar, Sam Dew, Baby Keem",46,86176890
928,California Breeze,Lil Baby,46,85559365
880,Baile no Morro,"Mc Vitin Da Igrejinha, MC Tairon, DJ Win",45,129314708
884,Caile,Luar La L,45,273914335
762,Love Yourself,Justin Bieber,44,2123309722
930,Casei Com a Putaria,"MC Ryan SP, Love Funk, Mc Paiva ZS",44,187701588



--- Top 10 Songs by Streams ---


,track_name,artist(s)_name,streams,streams
55,Blinding Lights,The Weeknd,3703895074,3703895074
179,Shape of You,Ed Sheeran,3562543890,3562543890
86,Someone You Loved,Lewis Capaldi,2887241814,2887241814
620,Dance Monkey,Tones and I,2864791672,2864791672
41,Sunflower - Spider-Man: Into the Spider-Verse,"Post Malone, Swae Lee",2808096550,2808096550
162,One Dance,"Drake, WizKid, Kyla",2713922350,2713922350
84,STAY (with Justin Bieber),"Justin Bieber, The Kid Laroi",2665343922,2665343922
140,Believer,Imagine Dragons,2594040133,2594040133
725,Closer,"The Chainsmokers, Halsey",2591224264,2591224264
48,Starboy,"The Weeknd, Daft Punk",2565529693,2565529693



--- Top 10 Songs by Valence % ---


,track_name,artist(s)_name,valence_%,streams
359,Zona De Perigo,Leo Santana,97,134294498
418,Doja,Central Cee,97,482257456
754,There's Nothing Holdin' Me Back,Shawn Mendes,97,1714490998
861,En El Radio Un Cochinero,Victor Cibrian,97,164856284
896,JGL,"Luis R Conriquez, La Adictiva",97,323455692
25,SABOR FRESA,Fuerza Regida,96,78300654
39,TQM,Fuerza Regida,96,176553476
53,(It Goes Like) Nanana - Edit,Peggy Gou,96,57876440
117,Rara Vez,"Taiu, Milo j",96,248088961
137,"Tere Vaaste (From ""Zara Hatke Zara Bachke"")","Sachin-Jigar, Shadab Faridi, Altamash Faridi, ...",96,54225632


# --- STAGE 3: EXHAUSTIVE STATISTICS & OUTLIER DETECTION ---


In [8]:
print("--- STAGE 3: DESCRIPTIVE STATISTICS & OUTLIERS ---")

# Set display option to suppress scientific notation for better readability
pd.options.display.float_format = '{:.2f}'.format

# 3.1 Descriptive Statistics
print("\nDescriptive Statistics for All Numerical Features:")
display(df_cleaned[numeric_cols].describe().T[['mean', '50%', 'std', 'min', 'max']])

# Reset display option to default after displaying, if necessary for other parts of the notebook
# pd.reset_option('display.float_format')

# 3.2 Automated Outlier Detection (IQR Method) for ALL numerical columns
print("\n[Automated Outlier Analysis - IQR Method]")
outlier_summary = []

for col in numeric_cols:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_cleaned[(df_cleaned[col] < lower_bound) | (df_cleaned[col] > upper_bound)]
    outlier_pct = (len(outliers) / len(df_cleaned)) * 100
    severity = "Low" if outlier_pct < 5 else "Medium" if outlier_pct < 10 else "High"

    outlier_summary.append({
        'Feature': col,
        'Outliers Count': len(outliers),
        'Percentage (%)': round(outlier_pct, 2),
        'Severity': severity
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outliers Count', ascending=False)
display(outlier_df[outlier_df['Outliers Count'] > 0].reset_index(drop=True))

--- STAGE 3: DESCRIPTIVE STATISTICS & OUTLIERS ---

Descriptive Statistics for All Numerical Features:


,mean,50%,std,min,max
acousticness_%,27.08,18.00,26.00,0.00,97.00
artist_count,1.56,1.00,0.89,1.00,8.00
bpm,122.55,121.00,28.07,65.00,206.00
danceability_%,66.98,69.00,14.63,23.00,96.00
energy_%,64.27,66.00,16.56,9.00,97.00
in_apple_charts,51.96,38.50,50.63,0.00,275.00
in_apple_playlists,67.87,34.00,86.47,0.00,672.00
in_deezer_charts,2.67,0.00,6.04,0.00,58.00
in_deezer_playlists,385.54,44.00,1131.08,0.00,12367.00
in_shazam_charts,56.91,2.00,157.51,0.00,1451.00



[Automated Outlier Analysis - IQR Method]


,Feature,Outliers Count,Percentage (%),Severity
0,in_deezer_playlists,154,16.18,High
1,released_year,150,15.76,High
2,in_shazam_charts,145,15.23,High
3,in_deezer_charts,143,15.02,High
4,speechiness_%,136,14.29,High
5,in_spotify_playlists,109,11.45,High
6,instrumentalness_%,87,9.14,Medium
7,in_apple_playlists,78,8.19,Medium
8,in_spotify_charts,78,8.19,Medium
9,streams,74,7.77,Medium


In [9]:
import pandas as pd
import numpy as np
import kagglehub
import os
import warnings
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Re-run Data Acquisition (from cell ebbb883a) if df_raw is not defined
if 'df_raw' not in globals():
    print("Re-acquiring data as 'df_raw' is not defined.")
    path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    df_raw = pd.read_csv(os.path.join(path, csv_file), encoding='latin-1')
else:
    print("'df_raw' already exists. Skipping re-acquisition.")

# Re-run Data Preprocessing (from cell 1190ecb2) if df_cleaned is not defined
if 'df_cleaned' not in globals():
    print("Re-processing data as 'df_cleaned' is not defined.")
    df_cleaned = df_raw.copy()

    # 2.1 Clean structural noise in 'streams'
    is_noise = pd.to_numeric(df_cleaned['streams'], errors='coerce').isna()
    if is_noise.sum() > 0: # Check if there was noise before printing to avoid false positives
        print(f"Found {is_noise.sum()} corrupted record(s) in 'streams' → Dropped (during re-processing)")
    df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['streams'])
    df_cleaned['streams'] = df_cleaned['streams'].astype(np.int64)

    # 2.2 Clean comma in chart columns
    platform_cols_to_clean = ['in_spotify_playlists','in_spotify_charts','in_apple_playlists',
                     'in_apple_charts','in_deezer_playlists','in_deezer_charts','in_shazam_charts']
    for col in platform_cols_to_clean:
        if col in df_cleaned.columns and df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].astype(str).str.replace(',', '')
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0) # Ensure it's numeric and handle NaNs

    # 2.3 Missing values
    if 'key' in df_cleaned.columns:
        df_cleaned['key'] = df_cleaned['key'].fillna('Unknown')
    if 'in_shazam_charts' in df_cleaned.columns:
        df_cleaned['in_shazam_charts'] = df_cleaned['in_shazam_charts'].fillna(0)

    # 2.4 Datetime + weekday
    if all(col in df_cleaned.columns for col in ['released_year','released_month','released_day']):
        df_cleaned['release_date'] = pd.to_datetime(
            df_cleaned[['released_year','released_month','released_day']].astype(str).agg('-'.join, axis=1), errors='coerce'
        )
        df_cleaned['release_day_name'] = df_cleaned['release_date'].dt.day_name()

    # 2.5 Feature classification (needed for other parts of the notebook potentially)
    exclude_for_classification = ['track_name', 'artist(s)_name', 'release_date']
    numeric_cols_temp = df_cleaned.select_dtypes(include=np.number).columns.difference(exclude_for_classification).tolist()
    categorical_cols_temp = df_cleaned.select_dtypes(exclude=np.number).columns.difference(exclude_for_classification).tolist()
    print(f"✅ Classified {len(numeric_cols_temp)} numerical + {len(categorical_cols_temp)} categorical features (during re-processing)")
else:
    print("'df_cleaned' already exists. Skipping re-processing.")

audio_features = ['bpm','danceability_%','valence_%','energy_%','acousticness_%','instrumentalness_%','liveness_%','speechiness_%']
stats = df_cleaned[audio_features].describe().T[['mean','50%','std']].round(2)
display(stats)

# Setup the grid for subplots with Plotly
n_rows = 2
n_cols = 4
fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=[f'Distribution of {col.replace("_", " ").title()}' for col in audio_features])

for i, col in enumerate(audio_features):
    row = (i // n_cols) + 1
    col_idx = (i % n_cols) + 1
    fig.add_trace(go.Histogram(x=df_cleaned[col], name=col.replace("_", " ").title(), marker_color='#1DB954'),
                  row=row, col=col_idx)
    fig.update_xaxes(title_text='', row=row, col=col_idx)
    fig.update_yaxes(title_text='Frequency', row=row, col=col_idx)

fig.update_layout(title_text='Numerical Audio Features Distribution (Plotly)', title_y=0.98,
                  height=800, showlegend=False, template='plotly_white')
fig.show()
save_div_to_json(fig, 'numerical_audio_features_distribution')

'df_raw' already exists. Skipping re-acquisition.
'df_cleaned' already exists. Skipping re-processing.


,mean,50%,std
bpm,122.55,121.00,28.07
danceability_%,66.98,69.00,14.63
valence_%,51.41,51.00,23.48
energy_%,64.27,66.00,16.56
acousticness_%,27.08,18.00,26.00
instrumentalness_%,1.58,0.00,8.41
liveness_%,18.21,12.00,13.72
speechiness_%,10.14,6.00,9.92


[saved JSON] eda_json/numerical_audio_features_distribution.json


In [10]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ensure df_cleaned is available (minimal re-creation) if not already defined
if 'df_cleaned' not in globals():
    print("Re-creating df_cleaned for cell 053d5c8f...")
    # For brevity, assuming df_raw exists or loading minimal subset if needed
    if 'df_raw' not in globals():
        import kagglehub
        import os
        path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
        csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
        df_raw = pd.read_csv(os.path.join(path, csv_file), encoding='latin-1')

    df_cleaned = df_raw.copy()
    df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['streams'])
    df_cleaned['streams'] = df_cleaned['streams'].astype(np.int64)

    platform_cols_to_clean = ['in_spotify_playlists','in_spotify_charts','in_apple_playlists',
                     'in_apple_charts','in_deezer_playlists','in_deezer_charts','in_shazam_charts']
    for col in platform_cols_to_clean:
        if col in df_cleaned.columns and df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].astype(str).str.replace(',', '')
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

    df_cleaned['key'] = df_cleaned['key'].fillna('Unknown')
    df_cleaned['in_shazam_charts'] = df_cleaned['in_shazam_charts'].fillna(0)

    if all(col in df_cleaned.columns for col in ['released_year','released_month','released_day']):
        df_cleaned['release_date'] = pd.to_datetime(
            df_cleaned[['released_year','released_month','released_day']].astype(str).agg('-'.join, axis=1), errors='coerce'
        )
        df_cleaned['release_day_name'] = df_cleaned['release_date'].dt.day_name()

    # Re-define numeric_cols and categorical_cols if needed
    exclude_for_classification = ['track_name', 'artist(s)_name', 'release_date']
    globals()['numeric_cols'] = df_cleaned.select_dtypes(include=np.number).columns.difference(exclude_for_classification).tolist()
    globals()['categorical_cols'] = df_cleaned.select_dtypes(exclude=np.number).columns.difference(exclude_for_classification).tolist()

print("=== 4.1 DISTRIBUTION  ===")

# Plot all Categorical Features (Bar Charts)
# Using categorical_cols defined in previous cells or above reconstruction
if 'categorical_cols' not in globals():
    exclude_for_classification = ['track_name', 'artist(s)_name', 'release_date']
    globals()['categorical_cols'] = df_cleaned.select_dtypes(exclude=np.number).columns.difference(exclude_for_classification).tolist()


for col in categorical_cols:
    # Limit to top 15 for readability as requested previously
    value_counts_cat = df_cleaned[col].value_counts().head(15).reset_index()
    value_counts_cat.columns = [col, 'Count']

    fig_cat = px.bar(value_counts_cat, x=col, y='Count',
                     title=f"Top 15 Distribution of {col.replace("_", " ").title()} (Plotly)",
                     color_discrete_sequence=['#1DB954'])
    fig_cat.update_xaxes(title_text=col.replace("_", " ").title(), categoryorder='total descending')
    fig_cat.update_yaxes(title_text='Count')
    fig_cat.show()
    save_div_to_json(fig_cat, f'categorical_distribution_{col.lower()}')


=== 4.1 DISTRIBUTION  ===


[saved JSON] eda_json/categorical_distribution_key.json


[saved JSON] eda_json/categorical_distribution_mode.json


[saved JSON] eda_json/categorical_distribution_release_day_name.json


# --- STAGE 4: BUILD TARGET VARIABLE & BASIC RELATIONSHIPS ---


In [11]:
# Bin streams thành Hit / Non-Hit
high_threshold = df_cleaned['streams'].quantile(0.75)
df_cleaned['is_hit'] = (df_cleaned['streams'] >= high_threshold).astype(int)

print(f"Hit threshold (Top 25%): {high_threshold:,.0f} streams")
print(df_cleaned['is_hit'].value_counts())

Hit threshold (Top 25%): 673,869,022 streams
is_hit
0    714
1    238
Name: count, dtype: int64


In [12]:
print("📋 Samples for Hit songs (Top 25%)")
display(df_cleaned[df_cleaned['is_hit']==1].sample(10)[['track_name','artist(s)_name','streams','danceability_%','energy_%']])

📋 Samples for Hit songs (Top 25%)


,track_name,artist(s)_name,streams,danceability_%,energy_%
440,Payphone,"Maroon 5, Wiz Khalifa",1479264469,74,74
693,Seï¿½ï¿½o,"Shawn Mendes, Camila Cabello",2484812918,76,52
510,Infinity,Jaymes Young,888046992,67,67
110,Money Trees,"Kendrick Lamar, Jay Rock",1093605526,74,53
608,Bored,Billie Eilish,777765388,60,33
568,Lost in the Fire,"The Weeknd, Gesaffelstein",686734357,66,68
588,happier,Olivia Rodrigo,850608354,39,45
640,'Till I Collapse,"Eminem, Nate Dogg",1695712020,55,85
395,Space Song,Beach House,789753877,51,79
754,There's Nothing Holdin' Me Back,Shawn Mendes,1714490998,86,80


# --- STAGE 5: THEMATIC DEEP DIVE ANALYSIS ---


## 5.1 ARTIST ANALYSIS


In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
# Removed seaborn and matplotlib imports as we are switching to Plotly
# from plotly.subplots import make_subplots # Not strictly needed if making separate figures

print("=== ARTIST IMPACT ANALYSIS ===")

# 1. Prepare data for Solo vs Collaboration
df_cleaned['is_collaboration'] = np.where(df_cleaned['artist_count'] > 1, 'Collaboration', 'Solo')
collab_stats = df_cleaned.groupby('is_collaboration')['streams'].mean().reset_index()

# Plot 1: Streams Distribution by Artist Count (Log Scale Boxplot)
fig1 = px.box(df_cleaned, x='artist_count', y='streams',
              title='Streams Distribution by Artist Count (Log Scale) (Plotly)',
              log_y=True, # Apply log scale to y-axis
              color_discrete_sequence=['#1DB954']) # Use a Spotify-like green color
fig1.update_layout(xaxis_title='Number of Artists', yaxis_title='Streams (Log Scale)')
fig1.show()
save_div_to_json(fig1, 'streams_distribution_by_artist_count_boxplot')


# Plot 2: Average Streams: Solo vs Collaboration (Bar Chart)
fig2 = px.bar(collab_stats, x='is_collaboration', y='streams',
              title='Average Streams: Solo vs Collaboration (Plotly)',
              color_discrete_sequence=['#1DB954'])
fig2.update_layout(xaxis_title='Type of Song', yaxis_title='Average Streams')
fig2.show()
save_div_to_json(fig2, 'average_streams_solo_vs_collaboration_bar')

=== ARTIST IMPACT ANALYSIS ===


[saved JSON] eda_json/streams_distribution_by_artist_count_boxplot.json


[saved JSON] eda_json/average_streams_solo_vs_collaboration_bar.json


In [14]:
import pandas as pd
import numpy as np
import kagglehub
import os
import warnings
import plotly.express as px # Ensure plotly.express is imported
warnings.filterwarnings('ignore') # Suppress warnings

# Ensure df_raw is available
if 'df_raw' not in globals():
    print("Re-loading df_raw for cell 7d08d576...")
    path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    df_raw = pd.read_csv(os.path.join(path, csv_file), encoding='latin-1')

# Ensure df_cleaned is available (minimal re-creation) if not already defined
if 'df_cleaned' not in globals():
    print("Re-creating df_cleaned for cell 7d08d576...")
    df_cleaned = df_raw.copy()
    df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['streams'])
    df_cleaned['streams'] = df_cleaned['streams'].astype(np.int64)

    platform_cols_to_clean = ['in_spotify_playlists','in_spotify_charts','in_apple_playlists',
                     'in_apple_charts','in_deezer_playlists','in_deezer_charts','in_shazam_charts']
    for col in platform_cols_to_clean:
        if col in df_cleaned.columns and df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].astype(str).str.replace(',', '')
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

    df_cleaned['key'] = df_cleaned['key'].fillna('Unknown')
    df_cleaned['in_shazam_charts'] = df_cleaned['in_shazam_charts'].fillna(0)

    # Re-create release_date and release_day_name if needed
    if all(col in df_cleaned.columns for col in ['released_year','released_month','released_day']):
        df_cleaned['release_date'] = pd.to_datetime(
            df_cleaned[['released_year','released_month','released_day']].astype(str).agg('-'.join, axis=1), errors='coerce'
        )
        df_cleaned['release_day_name'] = df_cleaned['release_date'].dt.day_name()

# 1. Prepare data
artist_count_dist = df_cleaned['artist_count'].value_counts().sort_index().reset_index()
artist_count_dist.columns = ['Artist Count', 'Number of Songs']

# 2. Plotting with Plotly Express
fig = px.bar(artist_count_dist, x='Artist Count', y='Number of Songs',
             title='Distribution of Songs by Artist Count (Plotly)',
             color_discrete_sequence=px.colors.sequential.Viridis)
fig.update_traces(texttemplate='%{y}', textposition='outside')
fig.update_layout(xaxis_title='Number of Artists', yaxis_title='Count of Songs')
fig.show()
save_div_to_json(fig, 'distribution_of_songs_by_artist_count')

[saved JSON] eda_json/distribution_of_songs_by_artist_count.json


In [15]:
import pandas as pd
import numpy as np
import kagglehub
import os
import warnings
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

# Ensure df_raw is available
if 'df_raw' not in globals():
    print("Re-loading df_raw for cell f5412e21...")
    path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    df_raw = pd.read_csv(os.path.join(path, csv_file), encoding='latin-1')

# Ensure df_cleaned is available (minimal re-creation) if not already defined
if 'df_cleaned' not in globals():
    print("Re-creating df_cleaned for cell f5412e21...")
    df_cleaned = df_raw.copy()
    df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['streams'])
    df_cleaned['streams'] = df_cleaned['streams'].astype(np.int64)

    platform_cols_to_clean = ['in_spotify_playlists','in_spotify_charts','in_apple_playlists',
                     'in_apple_charts','in_deezer_playlists','in_deezer_charts','in_shazam_charts']
    for col in platform_cols_to_clean:
        if col in df_cleaned.columns and df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].astype(str).str.replace(',', '')
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

    df_cleaned['key'] = df_cleaned['key'].fillna('Unknown')
    df_cleaned['in_shazam_charts'] = df_cleaned['in_shazam_charts'].fillna(0)

    if all(col in df_cleaned.columns for col in ['released_year','released_month','released_day']):
        df_cleaned['release_date'] = pd.to_datetime(
            df_cleaned[['released_year','released_month','released_day']].astype(str).agg('-'.join, axis=1), errors='coerce'
        )
        df_cleaned['release_day_name'] = df_cleaned['release_date'].dt.day_name()


print("🏆 TOP ARTISTS & SUPERSTAR EFFECT")

# Calculate total streams per artist
artist_streams = df_cleaned.groupby('artist(s)_name')['streams'].sum().reset_index()
artist_streams = artist_streams.sort_values('streams', ascending=False)

# Get Top 10
top10 = artist_streams.head(10).copy()
top10['percentage'] = (top10['streams'] / artist_streams['streams'].sum() * 100)

print(f"Top 10 artists account for {top10['percentage'].sum():.1f}% of total streams in the dataset.")

# Plotting with Plotly Express
fig = px.bar(top10, x='streams', y='artist(s)_name',
             title='Top 10 Artists by Total Streams (Superstar Effect)',
             orientation='h',
             color_discrete_sequence=['#1DB954'])

fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.update_xaxes(title_text='Total Streams')
fig.update_yaxes(title_text='Artist Name')

# Add percentage labels
for i, row in top10.iterrows():
    fig.add_annotation(x=row['streams'], y=row['artist(s)_name'],
                       text=f"{row['percentage']:.1f}%",
                       showarrow=False, xanchor='left', xshift=10)

fig.show()
save_div_to_json(fig, 'top_artists_by_streams')

# Display the table
display(top10[['artist(s)_name', 'streams', 'percentage']].round(2))


🏆 TOP ARTISTS & SUPERSTAR EFFECT
Top 10 artists account for 19.2% of total streams in the dataset.


[saved JSON] eda_json/top_artists_by_streams.json


,artist(s)_name,streams,percentage
571,The Weeknd,14185552870,2.90
557,Taylor Swift,14053658300,2.87
159,Ed Sheeran,13908947204,2.84
222,Harry Styles,11608645649,2.37
43,Bad Bunny,9997799607,2.04
430,Olivia Rodrigo,7442148916,1.52
170,Eminem,6183805596,1.26
75,Bruno Mars,5846920599,1.19
25,Arctic Monkeys,5569806731,1.14
228,Imagine Dragons,5272484650,1.08


### Unique Artists and Song Contributions

In [16]:
import pandas as pd
import numpy as np
import kagglehub
import os
import warnings
import plotly.express as px

warnings.filterwarnings('ignore')

# Ensure df_raw is available
if 'df_raw' not in globals():
    print("Re-loading df_raw for cell b4709d4f...")
    path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    df_raw = pd.read_csv(os.path.join(path, csv_file), encoding='latin-1')

# Ensure df_cleaned is available (minimal re-creation) if not already defined
if 'df_cleaned' not in globals():
    print("Re-creating df_cleaned for cell b4709d4f...")
    df_cleaned = df_raw.copy()
    df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['streams'])
    df_cleaned['streams'] = df_cleaned['streams'].astype(np.int64)

    platform_cols_to_clean = ['in_spotify_playlists','in_spotify_charts','in_apple_playlists',
                     'in_apple_charts','in_deezer_playlists','in_deezer_charts','in_shazam_charts']
    for col in platform_cols_to_clean:
        if col in df_cleaned.columns and df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].astype(str).str.replace(',', '')
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

    df_cleaned['key'] = df_cleaned['key'].fillna('Unknown')
    df_cleaned['in_shazam_charts'] = df_cleaned['in_shazam_charts'].fillna(0)

    if all(col in df_cleaned.columns for col in ['released_year','released_month','released_day']):
        df_cleaned['release_date'] = pd.to_datetime(
            df_cleaned[['released_year','released_month','released_day']].astype(str).agg('-'.join, axis=1), errors='coerce'
        )
        df_cleaned['release_day_name'] = df_cleaned['release_date'].dt.day_name()


print(f"Total unique artists: {df_cleaned['artist(s)_name'].nunique()}")

artist_song_counts = df_cleaned['artist(s)_name'].value_counts().reset_index()
artist_song_counts.columns = ['Artist', 'Number of Songs']

fig = px.histogram(artist_song_counts, x='Number of Songs', nbins=20,
                   title='Distribution of Songs per Artist (Plotly)',
                   labels={'Number of Songs': 'Number of Songs Released by Artist', 'count': 'Number of Artists'},
                   color_discrete_sequence=['#1DB954'])
fig.update_layout(xaxis_title_text='Number of Songs Released by Artist', yaxis_title_text='Number of Artists')
fig.show()
save_div_to_json(fig, 'distribution_of_songs_per_artist')

Total unique artists: 644


[saved JSON] eda_json/distribution_of_songs_per_artist.json


This analysis shows that the majority of artists have only one song in the top songs list, with a long tail of artists having multiple entries.

## 5.2 TEMPORAL ANALYSIS


In [17]:
import pandas as pd
import numpy as np
import kagglehub
import os
import warnings
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Ensure df_raw is available
if 'df_raw' not in globals():
    print("Re-loading df_raw for cell 55f06850...")
    path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    df_raw = pd.read_csv(os.path.join(path, csv_file), encoding='latin-1')

# Ensure df_cleaned is available (minimal re-creation) if not already defined
if 'df_cleaned' not in globals():
    print("Re-creating df_cleaned for cell 55f06850...")
    df_cleaned = df_raw.copy()
    df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['streams'])
    df_cleaned['streams'] = df_cleaned['streams'].astype(np.int64)

    platform_cols_to_clean = ['in_spotify_playlists','in_spotify_charts','in_apple_playlists',
                     'in_apple_charts','in_deezer_playlists','in_deezer_charts','in_shazam_charts']
    for col in platform_cols_to_clean:
        if col in df_cleaned.columns and df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].astype(str).str.replace(',', '')
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

    df_cleaned['key'] = df_cleaned['key'].fillna('Unknown')
    df_cleaned['in_shazam_charts'] = df_cleaned['in_shazam_charts'].fillna(0)

    if all(col in df_cleaned.columns for col in ['released_year','released_month','released_day']):
        df_cleaned['release_date'] = pd.to_datetime(
            df_cleaned[['released_year','released_month','released_day']].astype(str).agg('-'.join, axis=1), errors='coerce'
        )
        df_cleaned['release_day_name'] = df_cleaned['release_date'].dt.day_name()


print("\n=== TEMPORAL TRENDS ANALYSIS ===")

# 1. Clean dates and filter range
# Ensure release_date column is properly created if not already
if 'release_date' not in df_cleaned.columns:
    df_cleaned['release_date'] = pd.to_datetime(df_cleaned[['released_year', 'released_month', 'released_day']].astype(str).agg('-'.join, axis=1), errors='coerce')
df_temporal = df_cleaned.dropna(subset=['release_date'])
df_temporal = df_temporal[(df_temporal['released_year'] >= 2000) & (df_temporal['released_year'] <= 2023)]

# 2. Audio features evolution over years
all_audio_features = ['danceability_%', 'energy_%', 'valence_%', 'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%', 'bpm']
yearly_audio = df_temporal.groupby('released_year')[all_audio_features].mean().reset_index()

fig_trend = go.Figure()
for feature in all_audio_features:
    fig_trend.add_trace(go.Scatter(x=yearly_audio['released_year'], y=yearly_audio[feature], mode='lines+markers', name=feature.replace('_', ' ').title()))

fig_trend.update_layout(
    title_text='Music Features Trend Over Time (2000 - 2023) (Plotly)',
    xaxis_title='Year',
    yaxis_title='Average Percentage (%)',
    template='plotly_white',
    hovermode='x unified'
)
fig_trend.show()
save_div_to_json(fig_trend, 'music_features_trend_over_time')

# 3. Seasonal patterns (Songs Released by Month)
monthly_stats = df_temporal.groupby('released_month')['streams'].count().reset_index()
monthly_stats.columns = ['Month', 'Number of Songs']
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
# Ensure mapping for all months 1-12 if some are missing from data, fill with 0 then map
all_months = pd.DataFrame({'Month': range(1, 13)})
monthly_stats = pd.merge(all_months, monthly_stats, on='Month', how='left').fillna(0)
monthly_stats['Month Name'] = monthly_stats['Month'].apply(lambda x: month_names[int(x)-1])

fig_seasonal = px.bar(monthly_stats, x='Month Name', y='Number of Songs',
                      title='Seasonality: Number of Hits Released by Month (Plotly)',
                      color_discrete_sequence=['#1DB954'],
                      category_orders={"Month Name": month_names})
fig_seasonal.update_traces(texttemplate='%{y:.0f}', textposition='outside')
fig_seasonal.update_layout(xaxis_title='Month', yaxis_title='Number of Songs', template='plotly_white')
fig_seasonal.show()
save_div_to_json(fig_seasonal, 'seasonality_songs_by_month')



=== TEMPORAL TRENDS ANALYSIS ===


[saved JSON] eda_json/music_features_trend_over_time.json


[saved JSON] eda_json/seasonality_songs_by_month.json


In [18]:
import pandas as pd
import numpy as np
import kagglehub
import os
import warnings
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

# Ensure df_raw is available
if 'df_raw' not in globals():
    print("Re-loading df_raw for cell 6509427f...")
    path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    df_raw = pd.read_csv(os.path.join(path, csv_file), encoding='latin-1')

# Ensure df_cleaned is available (minimal re-creation) if not already defined
if 'df_cleaned' not in globals():
    print("Re-creating df_cleaned for cell 6509427f...")
    df_cleaned = df_raw.copy()
    df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['streams'])
    df_cleaned['streams'] = df_cleaned['streams'].astype(np.int64)

    platform_cols_to_clean = ['in_spotify_playlists','in_spotify_charts','in_apple_playlists',
                     'in_apple_charts','in_deezer_playlists','in_deezer_charts','in_shazam_charts']
    for col in platform_cols_to_clean:
        if col in df_cleaned.columns and df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].astype(str).str.replace(',', '')
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

    df_cleaned['key'] = df_cleaned['key'].fillna('Unknown')
    df_cleaned['in_shazam_charts'] = df_cleaned['in_shazam_charts'].fillna(0)

    if all(col in df_cleaned.columns for col in ['released_year','released_month','released_day']):
        df_cleaned['release_date'] = pd.to_datetime(
            df_cleaned[['released_year','released_month','released_day']].astype(str).agg('-'.join, axis=1), errors='coerce'
        )
        df_cleaned['release_day_name'] = df_cleaned['release_date'].dt.day_name()


print("\n=== TEMPORAL STRATEGY: THE FRIDAY PHENOMENON ===")

# Ensure release_date exists
if 'release_date' not in df_cleaned.columns:
    df_cleaned['release_date'] = pd.to_datetime(
        df_cleaned[['released_year', 'released_month', 'released_day']].astype(str).agg('-'.join, axis=1),
        errors='coerce'
    )

df_cleaned['day_of_week'] = df_cleaned['release_date'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_stats = df_cleaned['day_of_week'].value_counts().reindex(day_order).reset_index()
day_stats.columns = ['Day', 'Number of Releases']

# Create Plotly line plot
fig = px.line(day_stats, x='Day', y='Number of Releases',
              title='The Friday Phenomenon: Release Volume by Day of the Week (Plotly)',
              color_discrete_sequence=['#1DB954'],
              markers=True)

fig.update_layout(xaxis_title='Day of the Week', yaxis_title='Number of Releases')
fig.show()
save_div_to_json(fig, 'friday_phenomenon')


=== TEMPORAL STRATEGY: THE FRIDAY PHENOMENON ===


[saved JSON] eda_json/friday_phenomenon.json


## 5.3 PLATFORM ANALYSIS


In [19]:
import pandas as pd
import numpy as np
import kagglehub
import os
import warnings
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Ensure df_raw is available
if 'df_raw' not in globals():
    print("Re-loading df_raw for cell bce3c344...")
    path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    df_raw = pd.read_csv(os.path.join(path, csv_file), encoding='latin-1')

# Ensure df_cleaned is available (minimal re-creation) if not already defined
if 'df_cleaned' not in globals():
    print("Re-creating df_cleaned for cell bce3c344...")
    df_cleaned = df_raw.copy()
    df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['streams'])
    df_cleaned['streams'] = df_cleaned['streams'].astype(np.int64)

    platform_cols_to_clean = ['in_spotify_playlists','in_spotify_charts','in_apple_playlists',
                     'in_apple_charts','in_deezer_playlists','in_deezer_charts','in_shazam_charts']
    for col in platform_cols_to_clean:
        if col in df_cleaned.columns and df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].astype(str).str.replace(',', '')
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

    df_cleaned['key'] = df_cleaned['key'].fillna('Unknown')
    df_cleaned['in_shazam_charts'] = df_cleaned['in_shazam_charts'].fillna(0)

    if all(col in df_cleaned.columns for col in ['released_year','released_month','released_day']):
        df_cleaned['release_date'] = pd.to_datetime(
            df_cleaned[['released_year','released_month','released_day']].astype(str).agg('-'.join, axis=1), errors='coerce'
        )
        df_cleaned['release_day_name'] = df_cleaned['release_date'].dt.day_name()


print("\n=== PLATFORM COMPARISON ANALYSIS ===")

platform_cols = ['in_spotify_charts', 'in_apple_charts', 'in_deezer_charts', 'in_shazam_charts']
for col in platform_cols:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

platform_presence = (df_cleaned[platform_cols] > 0).astype(int)
platform_corr = platform_presence.corr()

fig_platform = make_subplots(rows=1, cols=2,
                             subplot_titles=('Cross-Platform Presence Correlation', 'Distribution of Songs by Number of Platforms Present'))

# 1. Heatmap
fig_platform.add_trace(go.Heatmap(
    z=platform_corr.values,
    x=platform_corr.columns,
    y=platform_corr.index,
    colorscale='Blues',
    colorbar=dict(title='Correlation')
), row=1, col=1)

for i, row in enumerate(platform_corr.values):
    for j, val in enumerate(row):
        fig_platform.add_annotation(
            x=platform_corr.columns[j], y=platform_corr.index[i],
            text=f'{val:.2f}', showarrow=False, font=dict(color='black' if val < 0.5 else 'white')
        )

# 2. Multi-platform distribution
platform_presence['platforms_count'] = platform_presence.sum(axis=1)
multi_platform_dist = platform_presence['platforms_count'].value_counts().sort_index().reset_index()
multi_platform_dist.columns = ['Number of Platforms', 'Number of Songs']

fig_platform.add_trace(go.Bar(
    x=multi_platform_dist['Number of Platforms'],
    y=multi_platform_dist['Number of Songs'],
    marker_color='#1DB954'
), row=1, col=2)
fig_platform.update_xaxes(title_text='Number of Platforms', row=1, col=2)
fig_platform.update_yaxes(title_text='Number of Songs', row=1, col=2)
fig_platform.update_traces(texttemplate='%{y}', textposition='outside', row=1, col=2)

fig_platform.update_layout(title_text='Platform Analysis (Plotly)', title_y=0.95,
                           height=600, showlegend=False, template='plotly_white')
fig_platform.show()
save_div_to_json(fig_platform, 'platform_analysis')


=== PLATFORM COMPARISON ANALYSIS ===


[saved JSON] eda_json/platform_analysis.json


In [20]:
import pandas as pd
import numpy as np
import kagglehub
import os
import warnings
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

# Ensure df_raw is available
if 'df_raw' not in globals():
    print("Re-loading df_raw for cell 2bc7da99...")
    path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    df_raw = pd.read_csv(os.path.join(path, csv_file), encoding='latin-1')

# Ensure df_cleaned is available (minimal re-creation) if not already defined
if 'df_cleaned' not in globals():
    print("Re-creating df_cleaned for cell 2bc7da99...")
    df_cleaned = df_raw.copy()
    df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['streams'])
    df_cleaned['streams'] = df_cleaned['streams'].astype(np.int64)

    platform_cols_to_clean = ['in_spotify_playlists','in_spotify_charts','in_apple_playlists',
                     'in_apple_charts','in_deezer_playlists','in_deezer_charts','in_shazam_charts']
    for col in platform_cols_to_clean:
        if col in df_cleaned.columns and df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].astype(str).str.replace(',', '')
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

    df_cleaned['key'] = df_cleaned['key'].fillna('Unknown')
    df_cleaned['in_shazam_charts'] = df_cleaned['in_shazam_charts'].fillna(0)

    if all(col in df_cleaned.columns for col in ['released_year','released_month','released_day']):
        df_cleaned['release_date'] = pd.to_datetime(
            df_cleaned[['released_year','released_month','released_day']].astype(str).agg('-'.join, axis=1), errors='coerce'
        )
        df_cleaned['release_day_name'] = df_cleaned['release_date'].dt.day_name()

print("\n=== CROSS-PLATFORM SUCCESS FACTORS ===")

platform_detailed = ['in_spotify_playlists', 'in_spotify_charts', 'in_apple_playlists',
                     'in_apple_charts', 'in_deezer_playlists', 'in_deezer_charts', 'in_shazam_charts']

# Ensure streams column is numeric before correlation calculation
df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
df_cleaned.dropna(subset=['streams'], inplace=True)

platform_stream_corr = df_cleaned[platform_detailed + ['streams']].corr()['streams'].drop('streams').sort_values()

# Create Plotly horizontal bar chart
fig = px.bar(platform_stream_corr.reset_index(),
             x=platform_stream_corr.values,
             y=platform_stream_corr.index,
             orientation='h',
             title='Correlation: Platform Presence vs Total Streams (Plotly)',
             color_discrete_sequence=['#1DB954'])

fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.update_xaxes(title_text='Correlation Coefficient')
fig.update_yaxes(title_text='Platform Metric')
fig.show()
save_div_to_json(fig, 'platform_stream_correlation')


=== CROSS-PLATFORM SUCCESS FACTORS ===


[saved JSON] eda_json/platform_stream_correlation.json


## 5.4 HIT PREDICTORS & AUDIO PROFILE


In [21]:
import pandas as pd
import numpy as np
import kagglehub
import os
import warnings
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

# Ensure df_raw is available
if 'df_raw' not in globals():
    print("Re-loading df_raw for cell 28574895...")
    path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    df_raw = pd.read_csv(os.path.join(path, csv_file), encoding='latin-1')

# Ensure df_cleaned is available (minimal re-creation) if not already defined
if 'df_cleaned' not in globals():
    print("Re-creating df_cleaned for cell 28574895...")
    df_cleaned = df_raw.copy()
    df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['streams'])
    df_cleaned['streams'] = df_cleaned['streams'].astype(np.int64)

    platform_cols_to_clean = ['in_spotify_playlists','in_spotify_charts','in_apple_playlists',
                     'in_apple_charts','in_deezer_playlists','in_deezer_charts','in_shazam_charts']
    for col in platform_cols_to_clean:
        if col in df_cleaned.columns and df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].astype(str).str.replace(',', '')
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

    df_cleaned['key'] = df_cleaned['key'].fillna('Unknown')
    df_cleaned['in_shazam_charts'] = df_cleaned['in_shazam_charts'].fillna(0)

    if all(col in df_cleaned.columns for col in ['released_year','released_month','released_day']):
        df_cleaned['release_date'] = pd.to_datetime(
            df_cleaned[['released_year','released_month','released_day']].astype(str).agg('-'.join, axis=1), errors='coerce'
        )
        df_cleaned['release_day_name'] = df_cleaned['release_date'].dt.day_name()

print("\n=== HIT PREDICTORS ANALYSIS ===")

# Ensure is_hit exists
if 'is_hit' not in df_cleaned.columns:
    high_threshold = df_cleaned['streams'].quantile(0.75)
    df_cleaned['is_hit'] = (df_cleaned['streams'] >= high_threshold).astype(int)

audio_cols = ['danceability_%', 'valence_%', 'energy_%', 'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%', 'bpm']
success_factors = audio_cols + ['artist_count', 'in_spotify_playlists', 'in_apple_playlists', 'in_deezer_playlists']

# Calculate correlation with is_hit
success_corr = df_cleaned[success_factors + ['is_hit']].corr()['is_hit'].drop('is_hit').sort_values()

# Create Plotly horizontal bar chart
fig = px.bar(success_corr.reset_index(),
             x=success_corr.values,
             y=success_corr.index,
             orientation='h',
             title='Hit Predictors: Factors Correlating with Top 25% Hit Status (Plotly)',
             color=['red' if x < 0 else '#1DB954' for x in success_corr.values])

fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.update_xaxes(title_text='Correlation with Hit Status')
fig.update_yaxes(title_text='Factor')
fig.show()
save_div_to_json(fig, 'hit_predictors_correlation')


=== HIT PREDICTORS ANALYSIS ===


[saved JSON] eda_json/hit_predictors_correlation.json


In [22]:
import pandas as pd
import numpy as np
import kagglehub
import os
import warnings
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots # Needed if combining multiple plots

warnings.filterwarnings('ignore')

# Ensure df_raw is available
if 'df_raw' not in globals():
    print("Re-loading df_raw for cell 80130d01...")
    path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    df_raw = pd.read_csv(os.path.join(path, csv_file), encoding='latin-1')

# Ensure df_cleaned is available (minimal re-creation) if not already defined
if 'df_cleaned' not in globals():
    print("Re-creating df_cleaned for cell 80130d01...")
    df_cleaned = df_raw.copy()
    df_cleaned['streams'] = pd.to_numeric(df_cleaned['streams'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['streams'])
    df_cleaned['streams'] = df_cleaned['streams'].astype(np.int64)

    platform_cols_to_clean = ['in_spotify_playlists','in_spotify_charts','in_apple_playlists',
                     'in_apple_charts','in_deezer_playlists','in_deezer_charts','in_shazam_charts']
    for col in platform_cols_to_clean:
        if col in df_cleaned.columns and df_cleaned[col].dtype == 'object':
            df_cleaned[col] = df_cleaned[col].astype(str).str.replace(',', '')
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').fillna(0)

    df_cleaned['key'] = df_cleaned['key'].fillna('Unknown')
    df_cleaned['in_shazam_charts'] = df_cleaned['in_shazam_charts'].fillna(0)

    if all(col in df_cleaned.columns for col in ['released_year','released_month','released_day']):
        df_cleaned['release_date'] = pd.to_datetime(
            df_cleaned[['released_year','released_month','released_day']].astype(str).agg('-'.join, axis=1), errors='coerce'
        )
        df_cleaned['release_day_name'] = df_cleaned['release_date'].dt.day_name()

# Ensure is_mega_hit exists for the radar chart and potentially for the scatter if filtering is needed
if 'is_mega_hit' not in df_cleaned.columns:
    high_stream_threshold = df_cleaned['streams'].quantile(0.75)
    df_cleaned['is_mega_hit'] = np.where(df_cleaned['streams'] >= high_stream_threshold, 'Mega Hit (Top 25%)', 'Normal Hit')
# Ensure is_hit also exists as it was used in prior cells to define high_threshold
if 'is_hit' not in df_cleaned.columns:
    high_threshold = df_cleaned['streams'].quantile(0.75)
    df_cleaned['is_hit'] = (df_cleaned['streams'] >= high_threshold).astype(int)

print("\n=== ADVANCED INSIGHTS: AUDIO PROFILE & MOOD ===")

# 1. THE \"MOOD\" OF 2023: Energy vs Valence (Plotly)
fig_mood = px.scatter(df_cleaned, x='valence_%', y='energy_%',
                      size='danceability_%', color='mode',
                      color_discrete_map={'Major': '#1DB954', 'Minor': '#E91E63'},
                      title="The Sonic Mood of 2023: Energy vs. Valence (Size = Danceability) (Plotly)",
                      labels={'valence_%': 'Valence (Sad -> Happy)', 'energy_%': 'Energy (Calm -> Intense)'})

fig_mood.add_hline(y=df_cleaned['energy_%'].mean(), line_dash="dash", line_color="gray",
                   annotation_text="Avg Energy", annotation_position="bottom right")
fig_mood.add_vline(x=df_cleaned['valence_%'].mean(), line_dash="dash", line_color="gray",
                   annotation_text="Avg Valence", annotation_position="top left")

fig_mood.update_layout(template='plotly_white')
fig_mood.show()
save_div_to_json(fig_mood, 'sonic_mood_scatter')

# 2. AUDIO FEATURE RADAR CHART: Mega Hit vs Normal (Plotly)
radar_cols = ['danceability_%', 'valence_%', 'energy_%', 'acousticness_%', 'liveness_%']
# Using already defined 'is_mega_hit' from above
radar_data = df_cleaned.groupby('is_mega_hit')[radar_cols].mean().reset_index()

# Prepare data for radar chart traces
categories = radar_cols
fig_radar = go.Figure()

# Mega Hit trace
fig_radar.add_trace(go.Scatterpolar(
    r=radar_data.loc[radar_data['is_mega_hit'] == 'Mega Hit (Top 25%)', radar_cols].values[0],
    theta=categories,
    fill='toself',
    name='Mega Hit (Top 25%)',
    line_color='#1DB954',
    opacity=0.6
))

# Normal Hit trace
fig_radar.add_trace(go.Scatterpolar(
    r=radar_data.loc[radar_data['is_mega_hit'] == 'Normal Hit', radar_cols].values[0],
    theta=categories,
    fill='toself',
    name='Normal Hit',
    line_color='#191414',
    opacity=0.6
))

fig_radar.update_layout(
    polar=dict(
        radialaxis_visible=True,
        radialaxis=dict(range=[0, 100])
    ),
    showlegend=True,
    title_text='Audio Profile Comparison: Mega Hits vs Normal Hits (Plotly)',
    template='plotly_white'
)
fig_radar.show()
save_div_to_json(fig_radar, 'audio_profile_radar')


=== ADVANCED INSIGHTS: AUDIO PROFILE & MOOD ===


[saved JSON] eda_json/sonic_mood_scatter.json


[saved JSON] eda_json/audio_profile_radar.json


In [23]:
print("\n=== ADVANCED INSIGHTS: SONIC MOOD (MATPLOTLIB) ===")
print("Note: The scatter plot 'The Sonic Mood of 2023: Energy vs. Valence' is now generated using Plotly in cell 80130d01, along with the radar chart.")


=== ADVANCED INSIGHTS: SONIC MOOD (MATPLOTLIB) ===
Note: The scatter plot 'The Sonic Mood of 2023: Energy vs. Valence' is now generated using Plotly in cell 80130d01, along with the radar chart.


# --- STAGE 6: EXECUTIVE SUMMARY ---


From this, we can see that **Solo tracks have a higher average number of streams** compared to collaborations, suggesting a 'solo artist effect' where individual artists tend to garner more attention per track in this dataset.

## 📝 EXECUTIVE SUMMARY OF FINDINGS (Rewritten)

### Dataset Characteristics

*   **Total Samples & Features**: The initial dataset contained 953 songs and 24 features. After initial cleaning, 952 samples remained due to one corrupted record in 'streams' being dropped.
*   **Missing Values (Initial Dataset)**:
    *   `key`: 95 missing values (9.97%). Handled by filling with 'Unknown' during preprocessing.
    *   `in_shazam_charts`: 50 missing values (5.25%). Handled by filling with 0 during preprocessing.
*   **Outliers (IQR Method) & Data Skewness**: Several numerical features exhibit a high number of 'outliers' according to the IQR method. It's important to note that for this dataset, these are often **characteristics of the data's inherent right-skewed distribution**, rather than errors:
    *   `in_deezer_playlists`, `in_spotify_playlists`, `in_shazam_charts`: These count-based features show a strong right-skew, with many songs having low or zero values, but a few 'super hits' possessing extremely high values (tens of thousands). This is typical for popularity metrics.
    *   `released_year`: While the majority of songs are recent (2020-2023), the presence of a few older tracks (1930-2000) causes them to be flagged as outliers by IQR, reflecting the dataset's diverse range rather than data errors.
    *   `speechiness_%`: Most songs have low speechiness (<15%), but some rap or spoken-word tracks naturally have very high values, appearing as outliers.
    *   Other features with notable outlier counts include `in_deezer_playlists` (154 outliers, 16.18%), `released_year` (150 outliers, 15.76%), `in_shazam_charts` (145 outliers, 15.23%), `in_deezer_charts` (143 outliers, 15.02%), `speechiness_%` (136 outliers, 14.29%), and `in_spotify_playlists` (109 outliers, 11.45%).
*   **Correlations**: Strong positive correlations exist between `streams` and playlist inclusions, notably with `in_spotify_playlists` (r=0.79) and `in_apple_playlists` (r=0.77), indicating their significant impact on a song's success. Audio features generally show low to moderate correlations.

Our comprehensive exploratory data analysis of the **Spotify Top Songs 2023** dataset reveals several key factors contributing to a song's success:

#### 1. The Dominance of Playlists and Strategic Distribution
*   **Playlist Power:** Inclusion in editorial playlists, particularly on Spotify and Apple, is by far the most significant predictor of a song's total streams and its status as a "Mega Hit." Correlations exceeding 0.77 highlight their crucial role in discovery and sustained listenership, far outweighing chart presence alone.
*   **Multi-platform Reach:** Highly successful songs often demonstrate a strong presence across multiple platforms (Spotify, Apple, Deezer, Shazam), indicating that broad distribution and visibility contribute to their overall impact.

#### 2. Evolving Audio Landscape and Hit Characteristics
*   **Upbeat and Energetic Core:** "Mega Hits" tend to exhibit higher mean danceability and energy. The sonic mood of 2023 leans towards tracks that are both energetic and positive (high valence).
*   **Temporal Shift in Production:** Historical trends (2000-2023) show a decline in 'acousticness' and a sustained or rising prominence of rhythmic and energetic features, reflecting a shift towards more modern, electronically-influenced pop.

#### 3. Artist Dynamics: Solo Power and Superstar Effect
*   **Solo Success:** Contrary to popular belief, solo artist tracks achieve significantly higher median and average stream counts compared to collaborations with multiple artists.
*   **Superstar Concentration:** The dataset clearly shows a "superstar effect," where a small number of top artists (e.g., Taylor Swift, The Weeknd, Ed Sheeran) command an overwhelmingly large percentage of the total streams, creating a highly skewed distribution.

#### 4. Strategic Release Timing
*   **The Friday Phenomenon:** Fridays are overwhelmingly the most popular day for new music releases, demonstrating a clear industry strategy to capitalize on "New Music Friday" trends.
*   **Key Release Windows:** Specific months like January and May show peaks in the number of hit songs released, suggesting strategic timing around new year resolutions and the lead-up to summer.

---
**Conclusion:** Achieving "Mega Hit" status in 2023 is less about a single intrinsic audio feature and more about a sophisticated interplay of **strategic playlist placements**, an **energetic and danceable audio profile**, and **optimized release timing**, often favoring solo artists in a market dominated by a few global superstars.